# BenePick RAG Colab Final

이 노트북은 Google Drive의 `MyDrive/BenePick/final_project-develope-colab-clean.zip`을 바로 가져와서 BenePick RAG를 코랩에서 실행하기 위한 최종본입니다.

실행 순서:
1. 가능하면 GPU 런타임으로 변경
2. 아래 셀을 위에서부터 순서대로 실행
3. `GROQ_API_KEY`는 셀 실행 시 숨김 입력으로 넣기


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

ZIP_PATH = Path('/content/drive/MyDrive/BenePick/final_project-develope-colab-clean.zip')
EXTRACT_ROOT = Path('/content/benepick_extract')

assert ZIP_PATH.exists(), f'ZIP not found: {ZIP_PATH}'
print('ZIP exists:', ZIP_PATH)


In [ ]:
%cd /content
!rm -rf /content/benepick_extract
!mkdir -p /content/benepick_extract
!unzip -q "/content/drive/MyDrive/BenePick/final_project-develope-colab-clean.zip" -d /content/benepick_extract

from pathlib import Path
import shutil

def is_project_root(path: Path) -> bool:
    return (path / 'requirements.txt').exists() and (path / 'rag').is_dir()

candidates = [p for p in [EXTRACT_ROOT] + list(EXTRACT_ROOT.rglob('*')) if p.is_dir() and is_project_root(p)]
assert candidates, 'Could not find project root containing requirements.txt and rag/'
PROJECT_DIR = min(candidates, key=lambda p: len(p.parts))
print('Detected PROJECT_DIR =', PROJECT_DIR)

for name in ['venv', '.git', '.claude']:
    target = PROJECT_DIR / name
    if target.exists():
        shutil.rmtree(target, ignore_errors=True)

print('Top-level files:')
for p in sorted(PROJECT_DIR.iterdir()):
    print(' -', p.name)


In [ ]:
import os

os.chdir(PROJECT_DIR)
print('cwd =', os.getcwd())
!python -V
!pip install -q -r requirements.txt


In [ ]:

import os
import torch
from getpass import getpass

# Korean experiment uses OpenAI only inside this Colab session.
os.environ['OPENAI_API_KEY'] = getpass('Enter your OpenAI API key: ')
os.environ['BENEPICK_CHROMA_PATH'] = str(PROJECT_DIR / 'chroma_db')
os.environ['BENEPICK_ENABLE_RERANKER'] = '1' if torch.cuda.is_available() else '0'

KOREAN_LLM_MODEL = 'gpt-4o-mini'   # RAG answer generator
KOREAN_RAGAS_MODEL = 'gpt-4.1'     # RAGAS judge/evaluator

print('CUDA available:', torch.cuda.is_available())
print('BENEPICK_ENABLE_RERANKER =', os.environ['BENEPICK_ENABLE_RERANKER'])
print('BENEPICK_CHROMA_PATH =', os.environ['BENEPICK_CHROMA_PATH'])
print('KOREAN_LLM_MODEL =', KOREAN_LLM_MODEL)
print('KOREAN_RAGAS_MODEL =', KOREAN_RAGAS_MODEL)


## Optional: Chroma 컬렉션 재구축

로그에 `Collection [benepick_policies] does not exist`가 보이면 이 셀들을 먼저 실행하세요. 성공하면 dense retrieval이 다시 활성화됩니다.


In [ ]:
import os
import sys

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

!python rag/rebuild_chroma.py


In [ ]:

import rag.pipeline as rag_pipeline
from langchain_openai import ChatOpenAI

# Reset cached searcher so Chroma/reranker settings from this Colab session are used.
rag_pipeline._searcher = None
rag_pipeline._searcher_mode = 'uninitialized'

# Swap only the experiment-session answer LLM to OpenAI.
rag_pipeline.llm = ChatOpenAI(
    model=KOREAN_LLM_MODEL,
    temperature=0.3,
    max_tokens=512,
)

print('RAG searcher cache reset complete')
print('Experiment answer LLM = OpenAI', KOREAN_LLM_MODEL)


In [ ]:
import os
import sys

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

from rag.pipeline import benepick_rag

result = benepick_rag(
    user_query='서울 청년 월세 지원 알려줘',
    lang_code='ko',
    user_condition={
        'region': '서울',
        'age': 27,
        'employment_status': 'UNEMPLOYED',
        'household_type': 'SINGLE',
    },
)

print('success =', result.get('success'))
result


In [ ]:
if result.get('success'):
    data = result.get('data', {})
    print('ANSWER')
    print('-' * 80)
    print(data.get('answer', ''))
    print('\nDOCS USED')
    print('-' * 80)
    for doc in data.get('docs_used', []):
        print(doc)
else:
    print(result)


## Batch Experiment To Excel

여러 질문을 한 번에 돌리고, 질문/답변/걸린 시간/점수/사용 문서를 엑셀(`.xlsx`)로 저장합니다.


In [ ]:
import json
import os
import sys
from pathlib import Path

PROJECT_DIR = Path(PROJECT_DIR)
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

# Korean natural-language experiment question set.
# Important: we intentionally avoid exact policy names so the test is not too easy.
# 30 broad real-user topics x 10 intents = 300 unique questions.
TOPICS = [
    '\uccad\ub144 \uc6d4\uc138',
    '\uccad\ub144 \uc804\uc138\uc790\uae08',
    '\uccad\ub144 \uc8fc\uac70\ube44',
    '\ubb34\uc8fc\ud0dd \uccad\ub144 \uc8fc\uac70',
    '1\uc778 \uac00\uad6c \uc8fc\uac70',
    '\ubbf8\ucde8\uc5c5 \uccad\ub144 \uc9c0\uc6d0\uae08',
    '\uccad\ub144 \uad6c\uc9c1\ud65c\ub3d9',
    '\uccad\ub144 \uba74\uc811\ube44',
    '\ucde8\uc5c5\uc900\ube44\uc0dd \uad50\uc721\ube44',
    '\uccad\ub144 \uc9c1\uc5c5\ud6c8\ub828',
    '\uccad\ub144 \ucc3d\uc5c5\uc790\uae08',
    '\uc608\ube44\ucc3d\uc5c5\uc790 \uc0ac\uc5c5\ud654\uc790\uae08',
    '\uccad\ub144 \uc18c\uc0c1\uacf5\uc778',
    '\uc790\uc601\uc5c5\uc790 \uc815\ucc45\uc790\uae08',
    '\uccad\ub144 \uc790\uc0b0\ud615\uc131',
    '\uccad\ub144 \ubaa9\ub3c8\ub9c8\ub828',
    '\ud559\uc790\uae08 \ub300\ucd9c \uc0c1\ud658',
    '\uc800\uc18c\ub4dd\uce35 \uc0dd\uacc4\ube44',
    '\uae34\uae09\ud55c \uc0dd\ud65c\ube44',
    '\uc8fc\uac70\uae09\uc5ec',
    '\uc758\ub8cc\ube44 \ubd80\ub2f4',
    '\uc815\uc2e0\uac74\uac15 \uc0c1\ub2f4',
    '\ucd9c\uc0b0 \uc9c0\uc6d0\uae08',
    '\uc721\uc544 \uc9c0\uc6d0\uae08',
    '\uc544\ub3d9 \uc591\uc721\ube44',
    '\ub178\uc778 \uae30\ucd08\uc5f0\uae08',
    '\ub178\uc778 \ub3cc\ubd04',
    '\uc7a5\uc560\uc778 \uc0dd\ud65c\uc9c0\uc6d0',
    '\uc7a5\uc560\uc778 \ud65c\ub3d9\uc9c0\uc6d0',
    '\ud55c\ubd80\ubaa8\uac00\uc871 \uc591\uc721\ube44',
]

INTENTS = [
    '\ubc1b\uc744 \uc218 \uc788\ub294 \uc870\uac74\uc774 \ubb50\uc608\uc694?',
    '\uc2e0\uccad \ubc29\ubc95\uc744 \uc54c\ub824\uc918',
    '\uc9c0\uc6d0 \uae08\uc561\uc774\ub098 \uae30\uac04\uc774 \uad81\uae08\ud574\uc694',
    '\uc628\ub77c\uc778\uc73c\ub85c \uc2e0\uccad\ud560 \uc218 \uc788\ub098\uc694?',
    '\ud544\uc694\ud55c \uc11c\ub958\uac00 \ubb50\uc608\uc694?',
    '\uc18c\ub4dd \uae30\uc900\uc774 \uc788\ub098\uc694?',
    '\uc11c\uc6b8\uc5d0 \uc0b4\uba74 \uc2e0\uccad\ud560 \uc218 \uc788\ub098\uc694?',
    '27\uc138 \ubb34\uc9c1 \uccad\ub144\ub3c4 \ub300\uc0c1\uc774 \ub420 \uc218 \uc788\ub098\uc694?',
    '\uc774\ub7f0 \uc0c1\ud669\uc5d0\uc11c \ud568\uaed8 \ubcfc \ub9cc\ud55c \uc815\ucc45\uc744 \ucd94\ucc9c\ud574\uc918',
    '\uc2e0\uccad\ud558\uae30 \uc804\uc5d0 \ud655\uc778\ud574\uc57c \ud560 \uc810\uc744 \uc815\ub9ac\ud574\uc918',
]

questions = []
seen_questions = set()
for topic in TOPICS:
    for intent in INTENTS:
        question = ' '.join(f'{topic} {intent}'.split())
        if question not in seen_questions:
            seen_questions.add(question)
            questions.append(question)

if len(questions) != 300:
    raise ValueError(f'Question set invalid: total={len(questions)}, unique={len(set(questions))}')

EXPERIMENT_QUESTIONS = questions

EXPERIMENT_USER_CONDITION = {
    'region': '\uc11c\uc6b8',
    'age': 27,
    'employment_status': 'UNEMPLOYED',
    'household_type': 'SINGLE',
}

EXPERIMENT_LANG_CODE = 'ko'
EXPERIMENT_OUTPUT_XLSX = '/content/drive/MyDrive/BenePick/benepick_experiment_results_ko_openai.xlsx'
EXPERIMENT_CHECKPOINT_XLSX = '/content/drive/MyDrive/BenePick/benepick_experiment_results_ko_openai_checkpoint.xlsx'

RUN_RAGAS = True
RAGAS_MAX_ROWS = 60

MIN_DELAY_BETWEEN_QUESTIONS = 0.8
RATE_LIMIT_MAX_RETRIES = 8
RATE_LIMIT_BASE_SLEEP = 1.5
CHECKPOINT_EVERY = 10
MAX_DOCS_FOR_ANSWER = 3

print('cwd =', PROJECT_DIR)
print('topics =', len(TOPICS))
print('questions =', len(EXPERIMENT_QUESTIONS))
print('unique =', len(set(EXPERIMENT_QUESTIONS)))
print('duplicates =', len(EXPERIMENT_QUESTIONS) - len(set(EXPERIMENT_QUESTIONS)))
print('user_condition =', EXPERIMENT_USER_CONDITION)
print('run_ragas =', RUN_RAGAS)
print('ragas_max_rows =', RAGAS_MAX_ROWS)
print('output_xlsx =', EXPERIMENT_OUTPUT_XLSX)
print('checkpoint_xlsx =', EXPERIMENT_CHECKPOINT_XLSX)
print('experiment_lang_code =', EXPERIMENT_LANG_CODE)
print('sample questions:')
for q in EXPERIMENT_QUESTIONS[:10]:
    print('-', q)


In [ ]:

import json
import os
import sys
import time
from datetime import datetime

import pandas as pd

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

from rag.pipeline import retrieve_rag_documents, generate_answer


def is_rate_limit_error(exc):
    text = str(exc).lower()
    return (
        'rate limit' in text
        or '429' in text
        or 'rate_limit_exceeded' in text
        or 'tokens per minute' in text
        or 'requests per minute' in text
        or 'rate_limit_error' in text
    )


def run_with_retry(fn, label='task'):
    for attempt in range(RATE_LIMIT_MAX_RETRIES + 1):
        try:
            return fn()
        except Exception as e:
            if (not is_rate_limit_error(e)) or (attempt >= RATE_LIMIT_MAX_RETRIES):
                raise

            wait = min(RATE_LIMIT_BASE_SLEEP * (2 ** attempt), 30.0)
            print(f'[{label}] rate limit -> retry in {wait:.2f}s ({attempt + 1}/{RATE_LIMIT_MAX_RETRIES})')
            time.sleep(wait)


rows = []

for idx, question in enumerate(EXPERIMENT_QUESTIONS, start=1):
    print(f'[{idx}/{len(EXPERIMENT_QUESTIONS)}] {question}')

    if idx > 1 and MIN_DELAY_BETWEEN_QUESTIONS > 0:
        time.sleep(MIN_DELAY_BETWEEN_QUESTIONS)

    run_started_at = time.perf_counter()

    try:
        retrieval = run_with_retry(
            lambda: retrieve_rag_documents(question, EXPERIMENT_USER_CONDITION),
            label='retrieve'
        )
        retrieval_success = bool(retrieval.get('success'))

        if retrieval_success:
            rdata = retrieval.get('data', {})
            final_docs = rdata.get('final_docs', [])
            answer_docs = final_docs[:MAX_DOCS_FOR_ANSWER]
            contexts = [d.get('evidence_text', '') for d in answer_docs if d.get('evidence_text')]

            answer_started_at = time.perf_counter()
            answer = run_with_retry(
                lambda: generate_answer(question, answer_docs, EXPERIMENT_LANG_CODE),
                label='generate'
            )
            answer_time_ms = round((time.perf_counter() - answer_started_at) * 1000)

            docs_used = rdata.get('docs_used', [])
            top1_score = docs_used[0].get('score') if docs_used else None
            avg_doc_score = (
                round(sum(d.get('score', 0) for d in docs_used) / len(docs_used), 4)
                if docs_used else None
            )

            rows.append({
                'question_no': idx,
                'question': question,
                'success': True,
                'answer': answer,
                'elapsed_sec': round(time.perf_counter() - run_started_at, 3),
                'search_time_ms': rdata.get('search_time_ms'),
                'rerank_time_ms': rdata.get('rerank_time_ms'),
                'crag_time_ms': rdata.get('crag_time_ms'),
                'answer_time_ms': answer_time_ms,
                'doc_count': rdata.get('doc_count', len(final_docs)),
                'top1_score': top1_score,
                'avg_doc_score': avg_doc_score,
                'top_policy_name': docs_used[0].get('policy_name') if docs_used else None,
                'docs_used_json': json.dumps(docs_used, ensure_ascii=False),
                'contexts_json': json.dumps(contexts, ensure_ascii=False),
                'contexts_list': contexts,
                'user_condition_json': json.dumps(EXPERIMENT_USER_CONDITION, ensure_ascii=False),
                'timestamp': retrieval.get('timestamp') or datetime.now().strftime('%Y-%m-%dT%H:%M:%S'),
            })
        else:
            rows.append({
                'question_no': idx,
                'question': question,
                'success': False,
                'answer': retrieval.get('error_message', ''),
                'elapsed_sec': round(time.perf_counter() - run_started_at, 3),
                'search_time_ms': None,
                'rerank_time_ms': None,
                'crag_time_ms': None,
                'answer_time_ms': None,
                'doc_count': None,
                'top1_score': None,
                'avg_doc_score': None,
                'top_policy_name': None,
                'docs_used_json': '[]',
                'contexts_json': '[]',
                'contexts_list': [],
                'user_condition_json': json.dumps(EXPERIMENT_USER_CONDITION, ensure_ascii=False),
                'timestamp': datetime.now().strftime('%Y-%m-%dT%H:%M:%S'),
            })

    except Exception as e:
        rows.append({
            'question_no': idx,
            'question': question,
            'success': False,
            'answer': f'ERROR: {type(e).__name__}: {e}',
            'elapsed_sec': round(time.perf_counter() - run_started_at, 3),
            'search_time_ms': None,
            'rerank_time_ms': None,
            'crag_time_ms': None,
            'answer_time_ms': None,
            'doc_count': None,
            'top1_score': None,
            'avg_doc_score': None,
            'top_policy_name': None,
            'docs_used_json': '[]',
            'contexts_json': '[]',
            'contexts_list': [],
            'user_condition_json': json.dumps(EXPERIMENT_USER_CONDITION, ensure_ascii=False),
            'timestamp': datetime.now().strftime('%Y-%m-%dT%H:%M:%S'),
        })

    if idx % CHECKPOINT_EVERY == 0:
        df_checkpoint = pd.DataFrame(rows)
        df_checkpoint.drop(columns=['contexts_list'], errors='ignore').to_excel(
            EXPERIMENT_CHECKPOINT_XLSX,
            index=False,
        )
        print(f'[checkpoint] saved -> {EXPERIMENT_CHECKPOINT_XLSX} ({idx} questions)')


df_results = pd.DataFrame(rows)
df_results


In [ ]:
import importlib
import subprocess
import sys
from datetime import datetime

from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter


def ensure_package(import_name: str, pip_name: str | None = None):
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name])


def safe_score(value):
    if isinstance(value, list):
        vals = [v for v in value if v is not None]
        if not vals:
            return None
        return float(sum(vals) / len(vals))
    if value is None:
        return None
    try:
        return float(value)
    except Exception:
        return None


df_results['ragas_faithfulness'] = None
df_results['ragas_answer_relevancy'] = None
df_results['ragas_average'] = None

if RUN_RAGAS:
    ensure_package('ragas')
    ensure_package('datasets')
    ensure_package('langchain_huggingface', 'langchain-huggingface')

    from datasets import Dataset
    from ragas import evaluate
    from ragas.metrics._faithfulness import Faithfulness
    from ragas.metrics._answer_relevance import AnswerRelevancy
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.run_config import RunConfig
    from langchain_huggingface import HuggingFaceEmbeddings
    from rag.pipeline import llm as pipeline_llm

    eval_df = df_results[(df_results['success'] == True) & (df_results['contexts_list'].map(lambda x: len(x) > 0))].copy()
    if RAGAS_MAX_ROWS is not None:
        eval_df = eval_df.head(RAGAS_MAX_ROWS)

    if len(eval_df) > 0:
        ragas_rows = [
            {
                'question': row['question'],
                'answer': row['answer'],
                'contexts': row['contexts_list'],
            }
            for _, row in eval_df.iterrows()
        ]

        try:
            ragas_dataset = Dataset.from_list(ragas_rows)
            ragas_llm = LangchainLLMWrapper(pipeline_llm)
            ragas_emb = LangchainEmbeddingsWrapper(
                HuggingFaceEmbeddings(
                    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
                    model_kwargs={'device': 'cpu'},
                )
            )

            ragas_result = evaluate(
                dataset=ragas_dataset,
                metrics=[Faithfulness(), AnswerRelevancy()],
                llm=ragas_llm,
                embeddings=ragas_emb,
                raise_exceptions=False,
                run_config=RunConfig(timeout=180, max_retries=5, max_workers=1),
            )

            ragas_df = ragas_result.to_pandas()
            for (orig_idx, _), (_, metric_row) in zip(eval_df.iterrows(), ragas_df.iterrows()):
                f = safe_score(metric_row.get('faithfulness'))
                a = safe_score(metric_row.get('answer_relevancy'))
                avg = round((f + a) / 2, 4) if f is not None and a is not None else None
                df_results.at[orig_idx, 'ragas_faithfulness'] = round(f, 4) if f is not None else None
                df_results.at[orig_idx, 'ragas_answer_relevancy'] = round(a, 4) if a is not None else None
                df_results.at[orig_idx, 'ragas_average'] = avg
        except Exception as e:
            print('RAGAS skipped due to error:', e)

summary = pd.DataFrame([
    {
        'total_questions': len(df_results),
        'success_count': int(df_results['success'].sum()),
        'fail_count': int((~df_results['success']).sum()),
        'avg_elapsed_sec': round(df_results['elapsed_sec'].mean(), 3),
        'avg_top1_score': round(df_results['top1_score'].dropna().mean(), 4) if df_results['top1_score'].notna().any() else None,
        'avg_doc_score': round(df_results['avg_doc_score'].dropna().mean(), 4) if df_results['avg_doc_score'].notna().any() else None,
        'avg_ragas_faithfulness': round(df_results['ragas_faithfulness'].dropna().astype(float).mean(), 4) if df_results['ragas_faithfulness'].notna().any() else None,
        'avg_ragas_answer_relevancy': round(df_results['ragas_answer_relevancy'].dropna().astype(float).mean(), 4) if df_results['ragas_answer_relevancy'].notna().any() else None,
        'avg_ragas_score': round(df_results['ragas_average'].dropna().astype(float).mean(), 4) if df_results['ragas_average'].notna().any() else None,
        'created_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    }
])

excel_df = df_results.drop(columns=['contexts_list'])

with pd.ExcelWriter(EXPERIMENT_OUTPUT_PATH, engine='openpyxl') as writer:
    excel_df.to_excel(writer, sheet_name='results', index=False)
    summary.to_excel(writer, sheet_name='summary', index=False)

    ws_results = writer.sheets['results']
    ws_summary = writer.sheets['summary']

    for ws in [ws_results, ws_summary]:
        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.alignment = Alignment(horizontal='center', vertical='center')

        for column_cells in ws.columns:
            max_len = max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells)
            ws.column_dimensions[get_column_letter(column_cells[0].column)].width = min(max(max_len + 2, 12), 80)

print('saved to:', EXPERIMENT_OUTPUT_PATH)
print('rows:', len(df_results), 'unique questions:', df_results['question'].nunique())
print('ragas enabled:', RUN_RAGAS, 'ragas_max_rows:', RAGAS_MAX_ROWS)



## Low-Score Questions (Top 20)

?? ?? ???? ??? ?? ?? 20?? ???? ?? ?? ??? ?????.


In [ ]:
import pandas as pd
from pathlib import Path

INPUT_XLSX = Path(EXPERIMENT_OUTPUT_XLSX)
OUTPUT_XLSX = INPUT_XLSX.with_name(INPUT_XLSX.stem + '_analysis.xlsx')

sheet_name = 'results'
df = pd.read_excel(INPUT_XLSX, sheet_name=sheet_name)

col_map = {
    'question': 'question' if 'question' in df.columns else ('??' if '??' in df.columns else None),
    'success': 'success' if 'success' in df.columns else ('????' if '????' in df.columns else None),
    'top1': 'top1_score' if 'top1_score' in df.columns else ('??1??' if '??1??' in df.columns else None),
    'avg_doc': 'avg_doc_score' if 'avg_doc_score' in df.columns else ('??????' if '??????' in df.columns else None),
    'ragas': 'ragas_average' if 'ragas_average' in df.columns else ('RAGAS_????' if 'RAGAS_????' in df.columns else None),
    'elapsed': 'elapsed_sec' if 'elapsed_sec' in df.columns else ('?????_?' if '?????_?' in df.columns else None),
    'answer': 'answer' if 'answer' in df.columns else ('??' if '??' in df.columns else None),
    'policy': 'top_policy_name' if 'top_policy_name' in df.columns else ('?????' if '?????' in df.columns else None),
}

if col_map['question'] is None or col_map['success'] is None:
    raise ValueError('??/???? ??? ?? ?????.')

success_series = df[col_map['success']]
if success_series.dtype == bool:
    success_mask = success_series
else:
    success_mask = success_series.astype(str).str.lower().isin(['true', '1', 'yes', 'y', 't', '??'])

work = df[success_mask].copy()

for new_col, src_col in [('top1_score_norm', col_map['top1']), ('avg_doc_score_norm', col_map['avg_doc']), ('ragas_score_norm', col_map['ragas'])]:
    work[new_col] = pd.to_numeric(work[src_col], errors='coerce') if src_col else pd.NA

weights = {'top1_score_norm': 0.4, 'avg_doc_score_norm': 0.2, 'ragas_score_norm': 0.4}
weighted_sum = pd.Series(0.0, index=work.index)
used_weight = pd.Series(0.0, index=work.index)

for c, w in weights.items():
    valid = work[c].notna()
    weighted_sum[valid] += work.loc[valid, c] * w
    used_weight[valid] += w

work['composite_score'] = weighted_sum / used_weight.replace(0, pd.NA)
work['score_gap'] = 1.0 - work['composite_score']

low20 = work.sort_values('composite_score', ascending=True).head(20).copy()

export_cols = [col_map['question']]
for c in [col_map['answer'], col_map['policy'], col_map['elapsed'], col_map['top1'], col_map['avg_doc'], col_map['ragas']]:
    if c:
        export_cols.append(c)
export_cols += ['composite_score', 'score_gap']

summary = pd.DataFrame([{
    'input_file': str(INPUT_XLSX),
    'total_rows': len(df),
    'success_rows': int(success_mask.sum()),
    'analyzed_rows': len(work),
    'avg_composite_score': round(work['composite_score'].mean(), 4) if work['composite_score'].notna().any() else None,
    'low20_avg_composite_score': round(low20['composite_score'].mean(), 4) if low20['composite_score'].notna().any() else None,
}])

with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as writer:
    summary.to_excel(writer, index=False, sheet_name='summary')
    low20[export_cols].to_excel(writer, index=False, sheet_name='low_score_top20')

print('saved ->', OUTPUT_XLSX)
summary


In [ ]:

import sys
import subprocess
import pandas as pd


def ensure_package(pkg_name, import_name=None):
    name = import_name or pkg_name
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg_name])


ragas_summary = {
    'ragas_enabled': RUN_RAGAS,
    'ragas_rows_evaluated': 0,
    'ragas_status': 'skipped',
    'faithfulness_mean': None,
    'answer_relevancy_mean': None,
    'ragas_average_mean': None,
}

if RUN_RAGAS:
    try:
        ensure_package('ragas')
        ensure_package('datasets')
        ensure_package('langchain-huggingface', 'langchain_huggingface')

        from datasets import Dataset
        from ragas import evaluate
        from ragas.metrics._faithfulness import Faithfulness
        from ragas.metrics._answer_relevance import AnswerRelevancy
        from ragas.llms import LangchainLLMWrapper
        from ragas.embeddings import LangchainEmbeddingsWrapper
        from ragas.run_config import RunConfig
        from langchain_openai import ChatOpenAI
        from langchain_huggingface import HuggingFaceEmbeddings

        eval_df = df_results[df_results['success'] == True].copy()

        if RAGAS_MAX_ROWS is not None:
            eval_df = eval_df.head(RAGAS_MAX_ROWS)

        eval_df = eval_df[
            eval_df['answer'].notna() &
            eval_df['question'].notna()
        ].copy()

        eval_df['contexts_list'] = eval_df['contexts_list'].apply(
            lambda x: x if isinstance(x, list) and len(x) > 0 else ['']
        )

        if len(eval_df) > 0:
            dataset = Dataset.from_pandas(
                eval_df[['question', 'answer', 'contexts_list']].rename(
                    columns={'contexts_list': 'contexts'}
                ),
                preserve_index=False,
            )

            ragas_llm = LangchainLLMWrapper(
                ChatOpenAI(
                    model=KOREAN_RAGAS_MODEL,
                    temperature=0,
                    max_tokens=512,
                )
            )
            ragas_emb = LangchainEmbeddingsWrapper(
                HuggingFaceEmbeddings(
                    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
                    model_kwargs={'device': 'cpu'},
                )
            )

            result = evaluate(
                dataset=dataset,
                metrics=[Faithfulness(), AnswerRelevancy()],
                llm=ragas_llm,
                embeddings=ragas_emb,
                raise_exceptions=False,
                run_config=RunConfig(timeout=180, max_retries=5, max_workers=1),
            )

            ragas_df = result.to_pandas().reset_index(drop=True)
            eval_df = eval_df.reset_index(drop=True)

            df_results.loc[eval_df.index, 'ragas_faithfulness'] = ragas_df['faithfulness']
            df_results.loc[eval_df.index, 'ragas_answer_relevancy'] = ragas_df['answer_relevancy']
            df_results.loc[eval_df.index, 'ragas_average'] = (
                ragas_df[['faithfulness', 'answer_relevancy']].mean(axis=1)
            )

            ragas_summary = {
                'ragas_enabled': True,
                'ragas_rows_evaluated': len(eval_df),
                'ragas_status': 'completed',
                'faithfulness_mean': round(df_results['ragas_faithfulness'].dropna().mean(), 4),
                'answer_relevancy_mean': round(df_results['ragas_answer_relevancy'].dropna().mean(), 4),
                'ragas_average_mean': round(df_results['ragas_average'].dropna().mean(), 4),
            }
        else:
            ragas_summary['ragas_status'] = 'no_valid_rows'

    except Exception as e:
        print('RAGAS skipped:', e)
        ragas_summary['ragas_status'] = f'failed: {type(e).__name__}: {e}'

summary_rows = [{
    'total_questions': len(df_results),
    'success_count': int(df_results['success'].fillna(False).sum()),
    'fail_count': int((~df_results['success'].fillna(False)).sum()),
    'avg_elapsed_sec': round(df_results['elapsed_sec'].dropna().mean(), 3) if len(df_results) else None,
    'avg_search_time_ms': round(df_results['search_time_ms'].dropna().mean(), 1) if df_results['search_time_ms'].notna().any() else None,
    'avg_rerank_time_ms': round(df_results['rerank_time_ms'].dropna().mean(), 1) if df_results['rerank_time_ms'].notna().any() else None,
    'avg_crag_time_ms': round(df_results['crag_time_ms'].dropna().mean(), 1) if df_results['crag_time_ms'].notna().any() else None,
    'avg_answer_time_ms': round(df_results['answer_time_ms'].dropna().mean(), 1) if df_results['answer_time_ms'].notna().any() else None,
    'avg_top1_score': round(df_results['top1_score'].dropna().mean(), 4) if df_results['top1_score'].notna().any() else None,
    'avg_doc_score': round(df_results['avg_doc_score'].dropna().mean(), 4) if df_results['avg_doc_score'].notna().any() else None,
    **ragas_summary,
}]

df_summary = pd.DataFrame(summary_rows)
df_results_to_save = df_results.drop(columns=['contexts_list'], errors='ignore').copy()

with pd.ExcelWriter(EXPERIMENT_OUTPUT_XLSX, engine='openpyxl') as writer:
    df_results_to_save.to_excel(writer, index=False, sheet_name='results')
    df_summary.to_excel(writer, index=False, sheet_name='summary')

    for ws_name, df in [('results', df_results_to_save), ('summary', df_summary)]:
        ws = writer.sheets[ws_name]
        for col_idx, col_name in enumerate(df.columns, start=1):
            max_len = max(
                len(str(col_name)),
                *(len(str(v)) for v in df[col_name].fillna('').astype(str).head(300))
            )
            width = min(max(max_len + 2, 12), 60)
            cell = ws.cell(row=1, column=col_idx)
            ws.column_dimensions[cell.column_letter].width = width

print('saved ->', EXPERIMENT_OUTPUT_XLSX)
print(df_summary)
df_summary


In [ ]:
import os
import sys
import time
import requests

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

!pkill -f "uvicorn.*rag.api:app" || true
!rm -f /content/rag_api.log
!nohup python -m uvicorn rag.api:app --app-dir "$PWD" --host 0.0.0.0 --port 8001 > /content/rag_api.log 2>&1 &

health_url = 'http://127.0.0.1:8001/health'
for attempt in range(30):
    try:
        res = requests.get(health_url, timeout=5)
        if res.ok:
            print('API server is ready:', res.text)
            break
    except Exception:
        pass
    time.sleep(2)
else:
    print('API server did not become ready. Showing last logs:')
    !tail -n 120 /content/rag_api.log
    raise RuntimeError('Failed to start FastAPI server on port 8001')

!tail -n 40 /content/rag_api.log


In [ ]:
import requests

payload = {
    'query': '서울 청년 월세 지원 알려줘',
    'lang_code': 'ko',
    'user_condition': {
        'region': '서울',
        'age': 27,
        'employment_status': 'UNEMPLOYED',
        'household_type': 'SINGLE',
    }
}

health = requests.get('http://127.0.0.1:8001/health', timeout=10)
print('health =', health.text)

res = requests.post('http://127.0.0.1:8001/rag/search', json=payload, timeout=180)
res.json()
